In [ ]:
import sys, os
sys.path.append(os.path.abspath("..")) # Allows us to import from grips and qokit directories
import matplotlib.pyplot as plt
import networkx as nx
import qokit.maxcut as mc
import numpy as np
from grips.QAOA_proxy_interface import QAOA_proxy, QAOA_proxy_expectation
from grips.QAOA_simulator import get_simulator, get_expectation, get_result
from grips.plot_utils import plot_heat_map
from grips.triangle_proxy import HardCodedTriangleProxy
from grips.paper_proxy import PaperProxy

# The following imports and seval statements make Julia proxy functions available
from juliacall import Main as jl
jl.seval('using Pkg')
jl.seval('Pkg.activate(joinpath(@__DIR__, "../julia"))')
jl.seval('Pkg.instantiate()')
jl.seval('include(joinpath(@__DIR__, "../julia/QAOA_proxy_interface.jl"))')
jl.seval('include(joinpath(@__DIR__, "../julia/paper_proxy.jl"))')
jl.seval('include(joinpath(@__DIR__, "../julia/triangle_proxy.jl"))')


# Define parameter ranges
gammas = np.linspace(0, np.pi, 50)
betas = np.linspace(0, np.pi, 50)
# Probabilities for the Erdos-Renyi graph generation
probabilities = [0.5]
# Whether to use python or julia implementations of the proxies
use_julia = True

'''
commented out the real QAOA parts for CPU use.
'''

def collect_parameter_data(G: nx.Graph, gammas: np.ndarray, betas: np.ndarray) -> np.ndarray:
    expectations = np.zeros([len(gammas), len(betas)])
    N = G.number_of_nodes()
    ising_model = mc.get_maxcut_terms(G)
    sim = get_simulator(N, ising_model)

    for i in range(len(gammas)):
        for j in range(len(betas)):
            gamma = np.array([gammas[i]])
            beta = np.array([betas[j]])
            result = get_result(N, ising_model, gamma, beta, sim)
            expectations[i][j] = get_expectation(N, ising_model, gamma, beta, sim, result)

    return expectations

def collect_parameter_data_proxy(proxy, gammas: np.ndarray, betas: np.ndarray) -> np.ndarray:
    expectations = np.zeros([len(gammas), len(betas)])
    for i in range(len(gammas)):
        for j in range(len(betas)):
            gamma = np.array([gammas[i]])
            beta = np.array([betas[j]])
            proxy_amplitudes = QAOA_proxy(proxy, gamma, beta)
            final_proxy_amplitudes = proxy_amplitudes[-1]
            expectation = QAOA_proxy_expectation(proxy, final_proxy_amplitudes)
            expectations[i][j] = expectation

    return expectations




for p in probabilities:
    for N in range(2, 7):
        G = nx.erdos_renyi_graph(N, p, seed=18) # generate graphs
        M = G.number_of_edges()

        # Define paths
        base_path = f"data_for_Expectation_Heatmaps/Erdős_Rényi/ER_p={p}"
        qaoa_path = os.path.join(base_path, f"QAOA_N={N}_M={M}.npz")
        paper_proxy_path = os.path.join(base_path, f"PaperProxy_N={N}_M={M}.npz")
        triangle_proxy_path = os.path.join(base_path, f"TriangleProxy_N={N}_M={M}.npz")

        # Create directories if they do not exist
        os.makedirs(base_path, exist_ok=True)

        if use_julia:
            # Julia Version
            triangle_proxy = jl.HardCodedTriangleProxy(M, N)
            paper_proxy = jl.PaperProxy(M, N, p)
        else:
            # Python version
            triangle_proxy = HardCodedTriangleProxy(M, N) # Will not be good for p != 0.5
            paper_proxy = PaperProxy(M, N, p)



        if os.path.exists(qaoa_path):
            data = np.load(qaoa_path)
            qaoa_expectations = data['expectations']
            gammas = data['gammas']
            betas = data['betas']
        else:
            qaoa_expectations = collect_parameter_data(G, gammas, betas)
            np.savez(qaoa_path, expectations=qaoa_expectations, gammas=gammas, betas=betas)

        if os.path.exists(paper_proxy_path):
            data = np.load(paper_proxy_path)
            paper_proxy_expectations = data['expectations']
            gammas = data['gammas']
            betas = data['betas']
        else:
            paper_proxy_expectations = collect_parameter_data_proxy(paper_proxy, gammas, betas)
            np.savez(paper_proxy_path, expectations=paper_proxy_expectations, gammas=gammas, betas=betas)

        if os.path.exists(triangle_proxy_path):
            data = np.load(triangle_proxy_path)
            triangle_proxy_expectations = data['expectations']
            gammas = data['gammas']
            betas = data['betas']
        else:
            triangle_proxy_expectations = collect_parameter_data_proxy(triangle_proxy, gammas, betas)
            np.savez(triangle_proxy_path, expectations=triangle_proxy_expectations, gammas=gammas, betas=betas)

         # Define image save paths
        img_base_path = f"Expectation_Heatmaps/Erdős_Rényi/ER_p={p}"
        qaoa_img_path = os.path.join(img_base_path, f"QAOA_N={N}_M={M}.png")
        paper_proxy_img_path = os.path.join(img_base_path, f"paper_proxy_N={N}_M={M}.png")
        triangle_proxy_img_path = os.path.join(img_base_path, f"triangle_proxy_N={N}_M={M}.png")

        # Create directories for images if they do not exist
        os.makedirs(img_base_path, exist_ok=True)

        # Generate and save heatmaps
        _ = plot_heat_map(gammas, betas, qaoa_expectations, f"True QAOA Expectation (N={N},M={M})", "Gamma", "Beta")
        plt.savefig(qaoa_img_path)
        _ = plot_heat_map(gammas, betas, paper_proxy_expectations, f"Expectation Proxies from Paper (N={N},M={M})", "Gamma", "Beta")
        plt.savefig(paper_proxy_img_path)
        _ = plot_heat_map(gammas, betas, triangle_proxy_expectations, f"Expectation Proxies from Us (Triangle) (N={N},M={M})", "Gamma", "Beta")
        plt.savefig(triangle_proxy_img_path)

        plt.show()

IndentationError: unindent does not match any outer indentation level (<string>, line 85)